In [1]:
import polars as pl
import matplotlib.pyplot as plt
import seaborn as sns
from elfen.util import normalize_column

import mantel
from itertools import combinations
import numpy as np
from sentence_transformers.util import cos_sim
from tqdm import tqdm

from src.utils import get_inf_feats, get_null_feats, get_nan_feats, symmetric_to_condensed

In [2]:
def get_conditions():
    conditions = [
        # INDIVIDUAL CONDITIONS
        # Human arguments
        (pl.col("model") == "Human"),
        # model, default (no context, no sociodemographic info, 
        # no explanation (info), no pros+cons)
        ((pl.col("context") == False),
         (pl.col("sociodemographic_info").is_null()),
         (pl.col("info").is_null()),
         (pl.col("pro").is_null()),
         (pl.col("contra").is_null())),
        # model, context only
        ((pl.col("model") != "Human"),
         (pl.col("context") == True),
         (pl.col("sociodemographic_info").is_null()),
         (pl.col("info").is_null()),
         (pl.col("pro").is_null()),
         (pl.col("contra").is_null())),
        # model, sociodemographic info only
        ((pl.col("model") != "Human"),
         (pl.col("context") == False),
         (pl.col("sociodemographic_info").is_not_null()),
         (pl.col("info").is_null()),
         (pl.col("pro").is_null()),
         (pl.col("contra").is_null())),
        # model, explanation (info) only
        ((pl.col("model") != "Human"),
         (pl.col("context") == False),
         (pl.col("sociodemographic_info").is_null()),
         (pl.col("info").is_not_null()),
         (pl.col("pro").is_null()),
         (pl.col("contra").is_null())),
        # model, pros+cons only
        ((pl.col("model") != "Human"),
         (pl.col("context") == False),
         (pl.col("sociodemographic_info").is_null()),
         (pl.col("info").is_null()),
         (pl.col("pro").is_not_null()),
         (pl.col("contra").is_not_null())),
        # TWO-WAY INTERACTIONS
        # model, context + sociodemographic info
        ((pl.col("model") != "Human"),
         (pl.col("context") == True),
         (pl.col("sociodemographic_info").is_not_null()),
         (pl.col("info").is_null()),
         (pl.col("pro").is_null()),
         (pl.col("contra").is_null())),
        # model, context + explanation (info)
        ((pl.col("model") != "Human"),
         (pl.col("context") == True),
         (pl.col("sociodemographic_info").is_null()),
         (pl.col("info").is_not_null()),
         (pl.col("pro").is_null()),
         (pl.col("contra").is_null())),
        # model, context + pros+cons
        ((pl.col("model") != "Human"),
         (pl.col("context") == True),
         (pl.col("sociodemographic_info").is_null()),
         (pl.col("info").is_null()),
         (pl.col("pro").is_not_null()),
         (pl.col("contra").is_not_null())),
        # model, sociodemographic info + explanation (info)
        ((pl.col("model") != "Human"),
         (pl.col("context") == False),
         (pl.col("sociodemographic_info").is_not_null()),
         (pl.col("info").is_not_null()),
         (pl.col("pro").is_null()),
         (pl.col("contra").is_null())),
        # model, sociodemographic info + pros+cons
        ((pl.col("model") != "Human"),
         (pl.col("context") == False),
         (pl.col("sociodemographic_info").is_not_null()),
         (pl.col("info").is_null()),
         (pl.col("pro").is_not_null()),
         (pl.col("contra").is_not_null())),
        # model, explanation (info) + pros+cons
        ((pl.col("model") != "Human"),
         (pl.col("context") == False),
         (pl.col("sociodemographic_info").is_null()),
         (pl.col("info").is_not_null()),
         (pl.col("pro").is_not_null()),
         (pl.col("contra").is_not_null())),
        # THREE-WAY INTERACTIONS
        # model, context + sociodemographic info + explanation (info)
        ((pl.col("model") != "Human"),
         (pl.col("context") == True),
         (pl.col("sociodemographic_info").is_not_null()),
         (pl.col("info").is_not_null()),
         (pl.col("pro").is_null()),
         (pl.col("contra").is_null())),
        # model, context + sociodemographic info + pros+cons
        ((pl.col("model") != "Human"),
         (pl.col("context") == True),
         (pl.col("sociodemographic_info").is_not_null()),
         (pl.col("info").is_null()),
         (pl.col("pro").is_not_null()),
         (pl.col("contra").is_not_null())),
        # model, context + explanation (info) + pros+cons
        ((pl.col("model") != "Human"),
         (pl.col("context") == True),
         (pl.col("sociodemographic_info").is_null()),
         (pl.col("info").is_not_null()),
         (pl.col("pro").is_not_null()),
         (pl.col("contra").is_not_null())),
        # model, sociodemographic info + explanation (info) + pros+cons
        ((pl.col("model") != "Human"),
         (pl.col("context") == False),
         (pl.col("sociodemographic_info").is_not_null()),
         (pl.col("info").is_not_null()),
         (pl.col("pro").is_not_null()),
         (pl.col("contra").is_not_null())),
        # ALL CONDITIONS
        # model, context + sociodemographic info + explanation (info) + pros+cons
        ((pl.col("model") != "Human"),
         (pl.col("context") == True),
         (pl.col("sociodemographic_info").is_not_null()),
         (pl.col("info").is_not_null()),
         (pl.col("pro").is_not_null()),
         (pl.col("contra").is_not_null()))
    ]
    condition_names = [
        "Human",
        "Default",
        "C",
        "SD",
        "Ex",
        "P&C",
        "C + SD",
        "C + Ex",
        "C + P&C",
        "SD + Ex",
        "SD + P&C",
        "Ex + P&C",
        "C + SD + Ex",
        "C + SD + P&C",
        "C + Ex + P&C",
        "SD + Ex + P&C",
        "C + SD + Ex + P&C"
    ]
    return conditions, condition_names

In [3]:
def add_conditions(df):
    conditions = df.with_columns(pl.lit("Dummy").alias("condition"))

    for condition, condition_name in zip(*get_conditions()):
        conditions = conditions.with_columns(
            pl.when(condition).then(pl.lit(condition_name))
            .otherwise(pl.col("condition")).alias("condition"))

    return conditions

# Load the data

In [4]:
prompts_de = pl.read_csv("prompts_de.csv")
prompts_fr = pl.read_csv("prompts_fr.csv")
prompts_it = pl.read_csv("prompts_it.csv")

In [5]:
prompts_de.head()

id,prompt,question,stance,sociodemographic_info,sociodemographic_group,info,pro,contra,context
str,str,str,str,str,str,str,str,str,bool
"""de_0""","""Sie gehören der folgenden sozi…","""Bei Ehepaaren ist die Höhe der…","""FAVOR""","""gender""","""Männlich""",null,null,null,false
"""de_1""","""Sie gehören der folgenden sozi…","""Bei Ehepaaren ist die Höhe der…","""FAVOR""","""age""","""18-34""",null,null,null,false
"""de_2""","""Sie gehören der folgenden sozi…","""Bei Ehepaaren ist die Höhe der…","""FAVOR""","""residence""","""Stadt""",null,null,null,false
"""de_3""","""Sie gehören der folgenden sozi…","""Bei Ehepaaren ist die Höhe der…","""FAVOR""","""civil_status""","""In Partnerschaft""",null,null,null,false
"""de_4""","""Sie gehören der folgenden sozi…","""Bei Ehepaaren ist die Höhe der…","""FAVOR""","""denomination""","""Evangelisch-reformiert""",null,null,null,false


In [6]:
corpus_de = pl.read_csv("output/processed/corpus_de_features.csv")
corpus_fr = pl.read_csv("output/processed/corpus_fr_features.csv")
corpus_it = pl.read_csv("output/processed/corpus_it_features.csv")

In [7]:
# get the question_id from the corpus and prompts dataframes to parallelize the dataframes later
question2prompts_de = prompts_de.join(corpus_de.select(["ID_question", "question"]), on="question", how="inner").select(["ID_question", "id"])
question2prompts_fr = prompts_fr.join(corpus_fr.select(["ID_question", "question"]), on="question", how="inner").select(["ID_question", "id"])
question2prompts_it = prompts_it.join(corpus_it.select(["ID_question", "question"]), on="question", how="inner").select(["ID_question", "id"])
# delete duplicate rows in question2prompts dataframes
question2prompts_de = question2prompts_de.unique()
question2prompts_fr = question2prompts_fr.unique()
question2prompts_it = question2prompts_it.unique()
# combine the question2prompts dataframes into one dataframe
question2prompts = pl.concat([question2prompts_de, question2prompts_fr, question2prompts_it], how="vertical")

In [8]:
# Load the data for LLaMA 4 Scout
df_de_llama4 = pl.read_csv("output/processed/llama4scout_de_features.csv"). \
    join(prompts_de, left_on="prompt_id", right_on="id", how="left"). \
    with_columns(pl.lit("llama_4_scout").alias("model"),
                 pl.lit("de").alias("language"))
df_fr_llama4 = pl.read_csv("output/processed/llama4scout_fr_features.csv"). \
    join(prompts_fr, left_on="prompt_id", right_on="id", how="left"). \
    with_columns(pl.lit("llama_4_scout").alias("model"),
                 pl.lit("fr").alias("language"))
df_it_llama4 = pl.read_csv("output/processed/llama4scout_it_features.csv"). \
    join(prompts_it, left_on="prompt_id", right_on="id", how="left"). \
    with_columns(pl.lit("llama_4_scout").alias("model"),
                 pl.lit("it").alias("language"))

# Load the data for GPT 4.1 Mini
df_de_gpt = pl.read_csv("output/processed/gpt-4.1-mini_de_features.csv"). \
    join(prompts_de, left_on="prompt_id", right_on="id", how="left"). \
    with_columns(pl.lit("gpt-4.1-mini").alias("model"),
                 pl.lit("de").alias("language"))
df_fr_gpt = pl.read_csv("output/processed/gpt-4.1-mini_fr_features.csv"). \
    join(prompts_fr, left_on="prompt_id", right_on="id", how="left"). \
    with_columns(pl.lit("gpt-4.1-mini").alias("model"),
                 pl.lit("fr").alias("language"))
df_it_gpt = pl.read_csv("output/processed/gpt-4.1-mini_it_features.csv"). \
    join(prompts_it, left_on="prompt_id", right_on="id", how="left"). \
    with_columns(pl.lit("gpt-4.1-mini").alias("model"),
                 pl.lit("it").alias("language"))

# Load the data for LLaMA 3.1-8B-Instruct
df_de_llama3 = pl.read_csv("output/processed/llama3.1-8b-instruct_de_features.csv"). \
    join(prompts_de, left_on="prompt_id", right_on="id", how="left"). \
    with_columns(pl.lit("llama_3.1-8b-instruct").alias("model"),
                 pl.lit("de").alias("language"))
df_fr_llama3 = pl.read_csv("output/processed/llama3.1-8b-instruct_fr_features.csv"). \
    join(prompts_fr, left_on="prompt_id", right_on="id", how="left"). \
    with_columns(pl.lit("llama_3.1-8b-instruct").alias("model"),
                 pl.lit("fr").alias("language"))
df_it_llama3 = pl.read_csv("output/processed/llama3.1-8b-instruct_it_features.csv"). \
    join(prompts_it, left_on="prompt_id", right_on="id", how="left"). \
    with_columns(pl.lit("llama_3.1-8b-instruct").alias("model"),
                 pl.lit("it").alias("language"))

# Load the data for Occiglot-7B-EU5 Instruct
df_de_occiglot = pl.read_csv("output/processed/occiglot-7b-eu5-instruct_de_features.csv"). \
    join(prompts_de, left_on="prompt_id", right_on="id", how="left"). \
    with_columns(pl.lit("occiglot-7b-eu5-instruct").alias("model"),
                 pl.lit("de").alias("language"))
df_fr_occiglot = pl.read_csv("output/processed/occiglot-7b-eu5-instruct_fr_features.csv"). \
    join(prompts_fr, left_on="prompt_id", right_on="id", how="left"). \
    with_columns(pl.lit("occiglot-7b-eu5-instruct").alias("model"),
                 pl.lit("fr").alias("language"))
df_it_occiglot = pl.read_csv("output/processed/occiglot-7b-eu5-instruct_it_features.csv"). \
    join(prompts_it, left_on="prompt_id", right_on="id", how="left"). \
    with_columns(pl.lit("occiglot-7b-eu5-instruct").alias("model"),
                 pl.lit("it").alias("language"))

In [9]:
# Get shared columns
shared_columns = set(df_de_llama4.columns) & set(df_fr_llama4.columns) & set(df_it_llama4.columns) & set(df_de_gpt.columns) & set(df_fr_gpt.columns) & set(df_it_gpt.columns) & set(df_de_llama3.columns) & set(df_fr_llama3.columns) & set(df_it_llama3.columns) & set(df_de_occiglot.columns) & set(df_fr_occiglot.columns) & set(df_it_occiglot.columns)

In [10]:
ordered_shared_columns = [col for col in df_de_llama4.columns if col in shared_columns]

In [11]:
# Combine all dataframes
df_generated = pl.concat([df_de_llama4.select(ordered_shared_columns),
                         df_fr_llama4.select(ordered_shared_columns),
                         df_it_llama4.select(ordered_shared_columns),
                         df_de_gpt.select(ordered_shared_columns),
                         df_fr_gpt.select(ordered_shared_columns),
                         df_it_gpt.select(ordered_shared_columns),
                         df_de_llama3.select(ordered_shared_columns),
                         df_fr_llama3.select(ordered_shared_columns),
                         df_it_llama3.select(ordered_shared_columns),
                         df_de_occiglot.select(ordered_shared_columns),
                         df_fr_occiglot.select(ordered_shared_columns),
                         df_it_occiglot.select(ordered_shared_columns)])

In [12]:
df_generated["sociodemographic_info"].n_unique()

8

In [13]:
for row in df_generated["sociodemographic_group"].value_counts().rows():
    print(row)

('Stadt', 4608)
('Andere Kirchen/Religionsgemeinschaften', 2280)
('Rechts und Liberal', 7056)
('18-34', 12792)
('Anderer Bildungsabschluss', 8184)
('35-49', 12072)
('Universität', 13848)
('Andere christliche Gemeinschaften', 8040)
('Verwitwet', 6960)
('Männlich', 13872)
('Römisch-katholisch', 13536)
('Konfessionslos', 12480)
('Links und Konservativ', 8760)
('Land', 13896)
(None, 13896)
('Mitte und Liberal', 7440)
('Jüdische Gemeinschaften', 744)
('In Partnerschaft', 8736)
('Berufsmatura', 6936)
('Islamische Gemeinschaften', 4584)
('Höhere Fachschule', 4992)
('Rechts und Konservativ', 3840)
('Mitte und Konservativ', 11184)
('Christ-katholisch', 8664)
('Höhere Berufsausbildung', 8688)
('Mitte und Konservativ-Liberal', 11208)
('Geschieden', 8304)
('Verheiratet', 13176)
('Fachhochschule', 9264)
('Ledig', 13323)
('Berufslehre oder Berufsschule', 8784)
('Links und Konservativ-Liberal', 12504)
('Sekundarschule', 4776)
('Anderes Geschlecht', 2232)
('Keine Schulbildung', 432)
('Weiblich', 10848

In [14]:
# recode sociodemographic features in the generated dataframe to match the corpus dataframe
df_generated = df_generated.with_columns(
    pl.when(pl.col("sociodemographic_info") == "gender")
    .then(pl.col("sociodemographic_group"))
    .otherwise(None).alias("gender"),
    pl.when(pl.col("sociodemographic_info") == "age")
    .then(pl.col("sociodemographic_group"))
    .otherwise(None).alias("age"),
    pl.when(pl.col("sociodemographic_info") == "education")
    .then(pl.col("sociodemographic_group"))
    .otherwise(None).alias("education"),
    pl.when(pl.col("sociodemographic_info") == "civil_status")
    .then(pl.col("sociodemographic_group"))
    .otherwise(None).alias("civil_status"),
    pl.when(pl.col("sociodemographic_info") == "denomination")
    .then(pl.col("sociodemographic_group"))
    .otherwise(None).alias("denomination"),
    pl.when(pl.col("sociodemographic_info") == "residence")
    .then(pl.col("sociodemographic_group"))
    .otherwise(None).alias("residence"),
    pl.when(pl.col("sociodemographic_info") == "political_spectrum")
    .then(pl.col("sociodemographic_group"))
    .otherwise(None).alias("political_spectrum")
)

In [15]:
# Join the generated dataframe with the question2prompts dataframe to get the question_id for each generated argument
df_generated = df_generated.join(question2prompts, left_on="prompt_id", right_on="id", how="left")

In [16]:
df_generated.head()

output,output_id,prompt_id,raw_sequence_length,n_tokens,n_sentences,tokens_per_sentence,n_characters,avg_word_length,n_types,n_long_words,n_lemmas,n_VERB_VerbForm_Fin,n_VERB_VerbForm_Inf,n_VERB_VerbForm_Part,n_VERB_Mood_Imp,n_VERB_Mood_Ind,n_VERB_Mood_Sub,n_VERB_Tense_Past,n_VERB_Tense_Pres,n_VERB_Person_1,n_VERB_Person_2,n_VERB_Person_3,n_VERB_Number_Plur,n_VERB_Number_Sing,n_NOUN_Gender_Fem,n_NOUN_Gender_Masc,n_NOUN_Number_Plur,n_NOUN_Number_Sing,n_PRON_PronType_Dem,n_PRON_PronType_Int,n_PRON_PronType_Prs,n_PRON_PronType_Rel,n_PRON_Gender_Fem,n_PRON_Gender_Masc,n_PRON_Number_Plur,n_PRON_Number_Sing,…,n_org,n_cardinal,n_date,n_gpe,n_person,n_money,n_product,n_time,n_percent,n_work_of_art,n_quantity,n_norp,n_loc,n_event,n_ordinal,n_fac,n_law,n_language,prompt,question,stance,sociodemographic_info,sociodemographic_group,info,pro,contra,context,model,language,gender,age,education,civil_status,denomination,residence,political_spectrum,ID_question
str,str,str,i64,i64,i64,f64,i64,f64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,…,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,str,str,str,str,str,str,str,str,bool,str,str,str,str,str,str,str,str,str,i64
"""Die Begrenzung der Rente auf 1…","""2025-07-17T17:23:00.495917136Z""","""de_0""",302,42,2,21.0,265,6.309524,37,22,32,0,1,2,0,0,0,0,0,0,0,0,0,0,7,1,4,6,0,0,0,0,0,0,0,0,…,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,"""Sie gehören der folgenden sozi…","""Bei Ehepaaren ist die Höhe der…","""FAVOR""","""gender""","""Männlich""",null,null,null,false,"""llama_4_scout""","""de""","""Männlich""",null,null,null,null,null,null,32216
"""Die Begrenzung der Rente für E…","""2025-07-17T17:23:01.835058734Z""","""de_0""",357,50,3,16.666667,313,6.26,40,26,36,1,4,1,0,1,0,0,1,0,0,1,0,1,6,2,5,7,0,0,0,1,0,0,1,0,…,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,"""Sie gehören der folgenden sozi…","""Bei Ehepaaren ist die Höhe der…","""FAVOR""","""gender""","""Männlich""",null,null,null,false,"""llama_4_scout""","""de""","""Männlich""",null,null,null,null,null,null,32216
"""Die Begrenzung der Rente auf 1…","""2025-07-17T17:23:03.555754915Z""","""de_0""",490,71,3,23.666667,427,6.014085,55,30,46,2,4,1,0,2,0,0,2,0,0,2,1,1,12,0,8,10,1,0,2,0,1,0,0,3,…,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,"""Sie gehören der folgenden sozi…","""Bei Ehepaaren ist die Höhe der…","""FAVOR""","""gender""","""Männlich""",null,null,null,false,"""llama_4_scout""","""de""","""Männlich""",null,null,null,null,null,null,32216
"""Die Begrenzung der Rentenhöhe …","""2025-07-17T17:23:04.951396543Z""","""de_1""",324,48,3,16.0,284,5.916667,41,22,38,2,1,3,0,2,0,0,2,0,0,2,1,1,7,2,4,6,0,0,0,1,0,0,2,0,…,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,"""Sie gehören der folgenden sozi…","""Bei Ehepaaren ist die Höhe der…","""FAVOR""","""age""","""18-34""",null,null,null,false,"""llama_4_scout""","""de""",null,"""18-34""",null,null,null,null,null,32216
"""Ich denke, die Begrenzung der …","""2025-07-17T17:23:06.180242153Z""","""de_1""",320,49,3,16.333333,279,5.693878,37,20,33,3,2,2,0,3,0,0,3,1,0,2,1,2,5,3,4,5,0,0,1,1,0,0,2,1,…,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,"""Sie gehören der folgenden sozi…","""Bei Ehepaaren ist die Höhe der…","""FAVOR""","""age""","""18-34""",null,null,null,false,"""llama_4_scout""","""de""",null,"""18-34""",null,null,null,null,null,32216


In [17]:
corpus_de["topic"].n_unique()

13

In [18]:
# Add model column to corpus dataframes
corpus_de = corpus_de.with_columns(pl.lit("Human").alias("model"))
corpus_fr = corpus_fr.with_columns(pl.lit("Human").alias("model"))
corpus_it = corpus_it.with_columns(pl.lit("Human").alias("model"))

# get shared columns between corpus dataframes
shared_columns_corpus = set(corpus_de.columns) & set(corpus_fr.columns) & set(corpus_it.columns)
ordered_shared_columns = [col for col in corpus_de.columns if col in shared_columns_corpus]

# Combine all dataframes
df_corpus = pl.concat([corpus_de.select(ordered_shared_columns),
                      corpus_fr.select(ordered_shared_columns),
                      corpus_it.select(ordered_shared_columns)])

# delete subcorpus dataframes to save memory
del df_de_llama4, df_fr_llama4, df_it_llama4, df_de_gpt, df_fr_gpt, df_it_gpt, df_de_llama3, df_fr_llama3, df_it_llama3, df_de_occiglot, df_fr_occiglot, df_it_occiglot, corpus_de, corpus_fr, corpus_it

In [19]:
df_corpus["topic"].n_unique()

13

In [20]:
# get the difference between the generated and corpus dataframes columns
diff_columns_corpus = set(df_generated.columns) - set(df_corpus.columns)
diff_columns_generated = set(df_corpus.columns) - set(df_generated.columns)
print(diff_columns_corpus, diff_columns_generated)

{'output_id', 'sociodemographic_info', 'prompt_id', 'n_PRON_Person_2', 'sociodemographic_group', 'output', 'prompt', 'context'} {'topic', 'argument_id', 'argument', 'n_intj'}


In [21]:
# Remove n_PRON_Person_2 from diff_columns_corpus as it is not a relevant feature for our analysis
diff_columns_corpus.remove("n_PRON_Person_2")
df_generated = df_generated.drop("n_PRON_Person_2")
# Remove n_intj from diff_columns_generated as it is not a relevant feature for our analysis
diff_columns_generated.remove("n_intj")
df_corpus = df_corpus.drop("n_intj")

In [22]:
# rename output_id to argument_id in the generated dataframe
df_generated = df_generated.rename({"output_id": "argument_id"})
# rename output to argument in the generated dataframe
df_generated = df_generated.rename({"output": "argument"})

In [23]:
# Remove unnecessary columns from the generated dataframe
df_generated = df_generated.drop([
    "prompt"
])
# add "context" column to the corpus dataframe with True as a dummy
df_corpus = df_corpus.with_columns(pl.lit(True).alias("context"))
# add sociodemographic_info column to the corpus dataframe with None as a dummy
df_corpus = df_corpus.with_columns(pl.lit(None).alias("sociodemographic_info"),
                                   pl.lit(None).alias("sociodemographic_group"))

In [24]:
# create ID_question2topic dictionary
id_question2topic = df_corpus.select(["ID_question", "topic"]).unique()
id_question2topic = {row[0]: row[1] for row in id_question2topic.iter_rows()}

# map the topic column in the generated dataframe using the ID_question2topic dictionary
df_generated = df_generated.with_columns(pl.col("ID_question").replace_strict(id_question2topic).alias("topic"))

In [25]:
# add dummy prompt_id column to the corpus dataframe
df_corpus = df_corpus.with_columns(pl.lit(None).alias("prompt_id"))

In [26]:
# shared columns between the generated and corpus dataframes
shared_columns = set(df_generated.columns) & set(df_corpus.columns)
ordered_shared_columns = [col for col in df_generated.columns if col in shared_columns]

In [27]:
len(ordered_shared_columns)

200

In [28]:
# get types per column for the generated and corpus dataframes
generated_types = df_generated.select(ordered_shared_columns).dtypes
corpus_types = df_corpus.select(ordered_shared_columns).dtypes
# print the name of the column where the type is different between the generated and corpus dataframes
for i, (gen_type, corp_type) in enumerate(zip(generated_types, corpus_types)):
    if gen_type != corp_type:
        print(f"Column {ordered_shared_columns[i]} has different types: generated={gen_type}, corpus={corp_type}")

Column argument_id has different types: generated=String, corpus=Int64
Column prompt_id has different types: generated=String, corpus=Null
Column sociodemographic_info has different types: generated=String, corpus=Null
Column sociodemographic_group has different types: generated=String, corpus=Null


In [29]:
df_generated = df_generated.with_columns(pl.col("argument_id").cast(pl.String))
df_corpus = df_corpus.with_columns(pl.col("argument_id").cast(pl.String))
df_corpus = df_corpus.with_columns(pl.col("sociodemographic_info").cast(pl.String),
                                   pl.col("sociodemographic_group").cast(pl.String))
df_corpus = df_corpus.with_columns(pl.col("prompt_id").cast(pl.String))

In [30]:
# Combine all dataframes
df = pl.concat([df_generated.select(ordered_shared_columns), df_corpus.select(ordered_shared_columns)], how="vertical")

In [31]:
df.describe()

statistic,argument,argument_id,prompt_id,raw_sequence_length,n_tokens,n_sentences,tokens_per_sentence,n_characters,avg_word_length,n_types,n_long_words,n_lemmas,n_VERB_VerbForm_Fin,n_VERB_VerbForm_Inf,n_VERB_VerbForm_Part,n_VERB_Mood_Imp,n_VERB_Mood_Ind,n_VERB_Mood_Sub,n_VERB_Tense_Past,n_VERB_Tense_Pres,n_VERB_Person_1,n_VERB_Person_2,n_VERB_Person_3,n_VERB_Number_Plur,n_VERB_Number_Sing,n_NOUN_Gender_Fem,n_NOUN_Gender_Masc,n_NOUN_Number_Plur,n_NOUN_Number_Sing,n_PRON_PronType_Dem,n_PRON_PronType_Int,n_PRON_PronType_Prs,n_PRON_PronType_Rel,n_PRON_Gender_Fem,n_PRON_Gender_Masc,n_PRON_Number_Plur,…,n_org,n_cardinal,n_date,n_gpe,n_person,n_money,n_product,n_time,n_percent,n_work_of_art,n_quantity,n_norp,n_loc,n_event,n_ordinal,n_fac,n_law,n_language,question,stance,sociodemographic_info,sociodemographic_group,info,pro,contra,context,model,language,gender,age,education,civil_status,denomination,residence,political_spectrum,ID_question,topic
str,str,str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,…,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,str,str,str,str,str,str,f64,str,str,str,str,str,str,str,str,str,f64,str
"""count""","""398729""","""398729""","""385971""",398729.0,398729.0,398729.0,398729.0,398729.0,398729.0,398729.0,398729.0,398729.0,398729.0,398729.0,398729.0,398729.0,398729.0,398729.0,398729.0,398729.0,398729.0,398729.0,398729.0,398729.0,398729.0,398729.0,398729.0,398729.0,398729.0,398729.0,398729.0,398729.0,398729.0,398729.0,398729.0,398729.0,…,398729.0,398729.0,398729.0,398729.0,398729.0,398729.0,398729.0,398729.0,398729.0,398729.0,398729.0,398729.0,398729.0,398729.0,398729.0,398729.0,398729.0,398729.0,"""398729""","""398729""","""372075""","""372075""","""176569""","""136168""","""136168""",398729.0,"""398729""","""398729""","""39710""","""60302""","""95438""","""72041""","""76454""","""31262""","""86156""",398729.0,"""398729"""
"""null_count""","""0""","""0""","""12758""",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,…,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,"""0""","""0""","""26654""","""26654""","""222160""","""262561""","""262561""",0.0,"""0""","""0""","""359019""","""338427""","""303291""","""326688""","""322275""","""367467""","""312573""",0.0,"""0"""
"""mean""",null,null,null,790.814395,133.085815,8.564491,NaN,679.6167,NaN,67.79844,50.954418,58.705241,4.656825,5.610964,2.076699,0.064114,3.910019,0.108813,1.093961,4.797296,0.950101,0.191797,3.545937,2.026359,3.689105,15.26432,9.142683,8.876944,17.471435,0.593197,0.025065,2.523029,0.901136,0.480707,0.821016,1.916354,…,0.580811,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.347243,0.0,0.0,0.0,0.0,0.0,null,null,null,null,null,null,null,0.516002,null,null,null,null,null,null,null,null,null,32249.312646,null
"""std""",null,null,null,655.252463,114.417852,12.258485,NaN,563.652023,NaN,38.896479,41.133593,32.14989,5.313642,5.720215,3.133303,0.649151,4.720949,0.505145,2.580368,5.31543,2.052177,1.334662,4.135821,3.226676,4.679835,13.439393,10.385991,8.472881,16.389406,1.365798,0.243541,6.032672,1.587625,1.236074,1.565164,4.506188,…,1.7089,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4.45015,0.0,0.0,0.0,0.0,0.0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,20.148834,null
"""min""","""""","""202300""","""de_0""",0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,…,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,"""Befürworten Sie die Einführung…","""AGAINST""","""age""","""18-34""","""""I sistemi di riconoscimento f…","""""Il referendum finanziario raf…",""" I sostenitori di tale divieto…",0.0,"""Human""","""de""","""Anderes Geschlecht""","""18-34""","""Anderer Bildungsabschluss""","""Geschieden""

In [32]:
df.filter(pl.col("model") == "Human").shape

(12758, 200)

In [33]:
df.columns

['argument',
 'argument_id',
 'prompt_id',
 'raw_sequence_length',
 'n_tokens',
 'n_sentences',
 'tokens_per_sentence',
 'n_characters',
 'avg_word_length',
 'n_types',
 'n_long_words',
 'n_lemmas',
 'n_VERB_VerbForm_Fin',
 'n_VERB_VerbForm_Inf',
 'n_VERB_VerbForm_Part',
 'n_VERB_Mood_Imp',
 'n_VERB_Mood_Ind',
 'n_VERB_Mood_Sub',
 'n_VERB_Tense_Past',
 'n_VERB_Tense_Pres',
 'n_VERB_Person_1',
 'n_VERB_Person_2',
 'n_VERB_Person_3',
 'n_VERB_Number_Plur',
 'n_VERB_Number_Sing',
 'n_NOUN_Gender_Fem',
 'n_NOUN_Gender_Masc',
 'n_NOUN_Number_Plur',
 'n_NOUN_Number_Sing',
 'n_PRON_PronType_Dem',
 'n_PRON_PronType_Int',
 'n_PRON_PronType_Prs',
 'n_PRON_PronType_Rel',
 'n_PRON_Gender_Fem',
 'n_PRON_Gender_Masc',
 'n_PRON_Number_Plur',
 'n_PRON_Number_Sing',
 'n_PRON_Person_1',
 'n_PRON_Person_3',
 'n_ADJ_Gender_Fem',
 'n_ADJ_Gender_Masc',
 'n_ADJ_Number_Plur',
 'n_ADJ_Number_Sing',
 'n_DET_Gender_Fem',
 'n_DET_Gender_Masc',
 'n_DET_Number_Plur',
 'n_DET_Number_Sing',
 'n_DET_Definite_Def',
 'n

In [34]:
feats = df.columns[3:166]

In [35]:
feats

['raw_sequence_length',
 'n_tokens',
 'n_sentences',
 'tokens_per_sentence',
 'n_characters',
 'avg_word_length',
 'n_types',
 'n_long_words',
 'n_lemmas',
 'n_VERB_VerbForm_Fin',
 'n_VERB_VerbForm_Inf',
 'n_VERB_VerbForm_Part',
 'n_VERB_Mood_Imp',
 'n_VERB_Mood_Ind',
 'n_VERB_Mood_Sub',
 'n_VERB_Tense_Past',
 'n_VERB_Tense_Pres',
 'n_VERB_Person_1',
 'n_VERB_Person_2',
 'n_VERB_Person_3',
 'n_VERB_Number_Plur',
 'n_VERB_Number_Sing',
 'n_NOUN_Gender_Fem',
 'n_NOUN_Gender_Masc',
 'n_NOUN_Number_Plur',
 'n_NOUN_Number_Sing',
 'n_PRON_PronType_Dem',
 'n_PRON_PronType_Int',
 'n_PRON_PronType_Prs',
 'n_PRON_PronType_Rel',
 'n_PRON_Gender_Fem',
 'n_PRON_Gender_Masc',
 'n_PRON_Number_Plur',
 'n_PRON_Number_Sing',
 'n_PRON_Person_1',
 'n_PRON_Person_3',
 'n_ADJ_Gender_Fem',
 'n_ADJ_Gender_Masc',
 'n_ADJ_Number_Plur',
 'n_ADJ_Number_Sing',
 'n_DET_Gender_Fem',
 'n_DET_Gender_Masc',
 'n_DET_Number_Plur',
 'n_DET_Number_Sing',
 'n_DET_Definite_Def',
 'n_DET_Definite_Ind',
 'tree_width',
 'tree_d

In [36]:
len(feats)

163

In [37]:
feats

['raw_sequence_length',
 'n_tokens',
 'n_sentences',
 'tokens_per_sentence',
 'n_characters',
 'avg_word_length',
 'n_types',
 'n_long_words',
 'n_lemmas',
 'n_VERB_VerbForm_Fin',
 'n_VERB_VerbForm_Inf',
 'n_VERB_VerbForm_Part',
 'n_VERB_Mood_Imp',
 'n_VERB_Mood_Ind',
 'n_VERB_Mood_Sub',
 'n_VERB_Tense_Past',
 'n_VERB_Tense_Pres',
 'n_VERB_Person_1',
 'n_VERB_Person_2',
 'n_VERB_Person_3',
 'n_VERB_Number_Plur',
 'n_VERB_Number_Sing',
 'n_NOUN_Gender_Fem',
 'n_NOUN_Gender_Masc',
 'n_NOUN_Number_Plur',
 'n_NOUN_Number_Sing',
 'n_PRON_PronType_Dem',
 'n_PRON_PronType_Int',
 'n_PRON_PronType_Prs',
 'n_PRON_PronType_Rel',
 'n_PRON_Gender_Fem',
 'n_PRON_Gender_Masc',
 'n_PRON_Number_Plur',
 'n_PRON_Number_Sing',
 'n_PRON_Person_1',
 'n_PRON_Person_3',
 'n_ADJ_Gender_Fem',
 'n_ADJ_Gender_Masc',
 'n_ADJ_Number_Plur',
 'n_ADJ_Number_Sing',
 'n_DET_Gender_Fem',
 'n_DET_Gender_Masc',
 'n_DET_Number_Plur',
 'n_DET_Number_Sing',
 'n_DET_Definite_Def',
 'n_DET_Definite_Ind',
 'tree_width',
 'tree_d

In [38]:
# remove redundant features (features that have the same value for all arguments)
redundant_feats = []
for feature in feats:
    if df.select(pl.col(feature).n_unique()).item() == 1:
        redundant_feats.append(feature)

df = df.drop(redundant_feats)
feats = [f for f in feats if f not in redundant_feats]

In [39]:
# removing empty arguments from the dataframe
df = df.filter(pl.col("raw_sequence_length") > 0)

In [40]:
get_inf_feats(df, feats)

{'dugast_u': 8579}

In [41]:
# fill dugast_u infs with a high value (e.g., 1000) to avoid issues with the analysis
df = df.with_columns(pl.when(pl.col("dugast_u") == float("inf"))
                     .then(1000)
                     .otherwise(pl.col("dugast_u"))
                     .alias("dugast_u"))

In [42]:
get_nan_feats(df, feats)

{'hdd': 33579}

In [43]:
df.drop("hdd")
feats.remove("hdd")

In [44]:
get_null_feats(df, feats)

{'avg_valence': 71,
 'avg_arousal': 71,
 'avg_dominance': 71,
 'avg_intensity_anger': 12276,
 'avg_intensity_anticipation': 7390,
 'avg_intensity_disgust': 15592,
 'avg_intensity_fear': 9256,
 'avg_intensity_joy': 6958,
 'avg_intensity_sadness': 11833,
 'avg_intensity_surprise': 15116,
 'avg_intensity_trust': 4042,
 'avg_concreteness': 7822,
 'avg_sd_concreteness': 7822}

In [45]:
# fill null values with the mean of the respective feature
for feat in get_null_feats(df, feats).keys():
    mean_value = df.select(pl.col(feat).mean()).item()
    df = df.with_columns(pl.when(pl.col(feat).is_null())
                         .then(mean_value)
                         .otherwise(pl.col(feat))
                         .alias(feat))

In [46]:
del df_corpus, df_generated, prompts_de, prompts_fr, prompts_it, question2prompts_de, question2prompts_fr, question2prompts_it

In [47]:
len(feats)

160

# Normalization of the data

In [47]:
# token normalize the features that start with "n_" but are not in the list of features to exclude from token normalization
replace_feats = [f for f in feats if f not in
                        ["n_tokens",
                         "n_types",
                         "n_sentences",
                         "n_characters",
                         "n_lemmas",
                         "n_syllables"] and
                        f.startswith("n_")]
for feature in replace_feats:
    df = df.with_columns(
        (pl.col(feature) / pl.col("n_tokens")). \
            alias(feature)
    )

In [48]:
# standardize all features (mean=0, std=1)
for feature in feats:
    df = normalize_column(df, feature)

In [49]:
df.describe()

statistic,argument,argument_id,prompt_id,raw_sequence_length,n_tokens,n_sentences,tokens_per_sentence,n_characters,avg_word_length,n_types,n_long_words,n_lemmas,n_VERB_VerbForm_Fin,n_VERB_VerbForm_Inf,n_VERB_VerbForm_Part,n_VERB_Mood_Imp,n_VERB_Mood_Ind,n_VERB_Mood_Sub,n_VERB_Tense_Past,n_VERB_Tense_Pres,n_VERB_Person_1,n_VERB_Person_2,n_VERB_Person_3,n_VERB_Number_Plur,n_VERB_Number_Sing,n_NOUN_Gender_Fem,n_NOUN_Gender_Masc,n_NOUN_Number_Plur,n_NOUN_Number_Sing,n_PRON_PronType_Dem,n_PRON_PronType_Int,n_PRON_PronType_Prs,n_PRON_PronType_Rel,n_PRON_Gender_Fem,n_PRON_Gender_Masc,n_PRON_Number_Plur,…,n_low_synsets_adv,n_entities,n_org,n_gpe,n_person,n_money,n_product,n_time,n_percent,n_work_of_art,n_quantity,n_norp,n_loc,n_event,n_ordinal,n_fac,n_law,n_language,question,stance,sociodemographic_info,sociodemographic_group,info,pro,contra,context,model,language,gender,age,education,civil_status,denomination,residence,political_spectrum,ID_question,topic
str,str,str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,…,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,str,str,str,str,str,str,f64,str,str,str,str,str,str,str,str,str,f64,str
"""count""","""388220""","""388220""","""375462""",388220.0,388220.0,388220.0,388220.0,388220.0,388220.0,388220.0,388220.0,388220.0,388220.0,388220.0,388220.0,388220.0,388220.0,388220.0,388220.0,388220.0,388220.0,388220.0,388220.0,388220.0,388220.0,388220.0,388220.0,388220.0,388220.0,388220.0,388220.0,388220.0,388220.0,388220.0,388220.0,388220.0,…,388220.0,388220.0,388220.0,388220.0,388220.0,388220.0,388220.0,388220.0,388220.0,388220.0,388220.0,388220.0,388220.0,388220.0,388220.0,388220.0,388220.0,388220.0,"""388220""","""388220""","""361822""","""361822""","""168722""","""129038""","""129038""",388220.0,"""388220""","""388220""","""39051""","""59202""","""92881""","""70388""","""74708""","""30728""","""84152""",388220.0,"""388220"""
"""null_count""","""0""","""0""","""12758""",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,…,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,"""0""","""0""","""26398""","""26398""","""219498""","""259182""","""259182""",0.0,"""0""","""0""","""349169""","""329018""","""295339""","""317832""","""313512""","""357492""","""304068""",0.0,"""0"""
"""mean""",null,null,null,-1.9115e-16,-4.0192e-17,5.6372e-18,-5.8393e-16,3.7923e-17,1.2584e-15,1.4928e-16,6.5604e-15,4.6708e-17,-1.0553e-15,3.2300e-16,2.7454e-18,1.1521e-17,-2.4931e-15,2.3795e-16,-3.8743e-16,-1.0079e-15,3.0071e-16,6.0591e-17,-2.5038e-17,-1.1136e-15,2.0149e-15,2.6960e-15,-2.9415e-15,-3.1041e-17,1.4980e-15,6.1870e-16,1.8508e-17,3.9570e-16,-9.4291e-16,8.7821e-16,-1.1603e-15,3.1426e-16,…,-6.4766e-16,3.6594e-16,1.4472e-16,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.383713,0.0,0.0,0.0,0.0,0.0,null,null,null,null,null,null,null,0.516045,null,null,null,null,null,null,null,null,null,32249.504108,null
"""std""",null,null,null,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,…,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4.504382,0.0,0.0,0.0,0.0,0.0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,20.175383,null
"""min""",""" ""","""202300""","""de_0""",-1.246425,-1.192205,-0.631745,-2.073476,-1.244941,-1.331794,-1.817455,-3.530497,-1.907983,-1.481626,-1.606196,-0.909567,-0.088682,-1.279825,-0.232455,-0.549852,-1.47543,-0.62804,-0.14845,-1.300616,-0.858245,-1.255052,-2.525951,-1.710221,-1.962812,-2.650146,-0.491953,-0.077206,-0.616245,-0.697413,-0.428613,-0.577276,-0.587751,…,-0.926621,-0.556488,-0.38352,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,"""Befürworten Sie die Einführung…","""AGAINST""","""age""","""18-34""","""""I sistemi di riconoscimento f…","""""Il referendum finanzia

# Add prompting conditions

In [50]:
df = add_conditions(df)

In [51]:
df.head()

argument,argument_id,prompt_id,raw_sequence_length,n_tokens,n_sentences,tokens_per_sentence,n_characters,avg_word_length,n_types,n_long_words,n_lemmas,n_VERB_VerbForm_Fin,n_VERB_VerbForm_Inf,n_VERB_VerbForm_Part,n_VERB_Mood_Imp,n_VERB_Mood_Ind,n_VERB_Mood_Sub,n_VERB_Tense_Past,n_VERB_Tense_Pres,n_VERB_Person_1,n_VERB_Person_2,n_VERB_Person_3,n_VERB_Number_Plur,n_VERB_Number_Sing,n_NOUN_Gender_Fem,n_NOUN_Gender_Masc,n_NOUN_Number_Plur,n_NOUN_Number_Sing,n_PRON_PronType_Dem,n_PRON_PronType_Int,n_PRON_PronType_Prs,n_PRON_PronType_Rel,n_PRON_Gender_Fem,n_PRON_Gender_Masc,n_PRON_Number_Plur,n_PRON_Number_Sing,…,n_entities,n_org,n_gpe,n_person,n_money,n_product,n_time,n_percent,n_work_of_art,n_quantity,n_norp,n_loc,n_event,n_ordinal,n_fac,n_law,n_language,question,stance,sociodemographic_info,sociodemographic_group,info,pro,contra,context,model,language,gender,age,education,civil_status,denomination,residence,political_spectrum,ID_question,topic,condition
str,str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,…,f64,f64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,str,str,str,str,str,str,str,bool,str,str,str,str,str,str,str,str,str,i64,str,str
"""Die Begrenzung der Rente auf 1…","""2025-07-17T17:23:00.495917136Z""","""de_0""",-0.783945,-0.831965,-0.550714,0.237451,-0.773409,0.336463,-0.864157,1.148557,-0.910461,-1.481626,-0.70375,2.007832,-0.088682,-1.279825,-0.232455,-0.549852,-1.47543,-0.62804,-0.14845,-1.300616,-0.858245,-1.255052,1.108304,-1.104256,0.74926,0.284128,-0.491953,-0.077206,-0.616245,-0.697413,-0.428613,-0.577276,-0.587751,-1.046768,…,1.170055,-0.38352,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,"""Bei Ehepaaren ist die Höhe der…","""FAVOR""","""gender""","""Männlich""",null,null,null,false,"""llama_4_scout""","""de""","""Männlich""",null,null,null,null,null,null,32216,"""Welfare state & family""","""SD"""
"""Die Begrenzung der Rente für E…","""2025-07-17T17:23:01.835058734Z""","""de_0""",-0.699438,-0.761674,-0.469683,-0.26325,-0.687675,0.320903,-0.784716,1.114528,-0.781748,-0.63706,1.426023,0.315741,-0.088682,-0.415415,-0.232455,-0.549852,-0.674269,-0.62804,-0.14845,-0.373372,-0.858245,-0.360005,0.090713,-0.692199,0.884863,0.225443,-0.491953,-0.077206,-0.616245,1.480823,-0.428613,-0.577276,0.508981,-1.046768,…,1.37724,-0.38352,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,"""Bei Ehepaaren ist die Höhe der…","""FAVOR""","""gender""","""Männlich""",null,null,null,false,"""llama_4_scout""","""de""","""Männlich""",null,null,null,null,null,null,32216,"""Welfare state & family""","""SD"""
"""Die Begrenzung der Rente auf 1…","""2025-07-17T17:23:03.555754915Z""","""de_0""",-0.495087,-0.577161,-0.469683,0.545575,-0.484059,0.243636,-0.387508,0.2439,-0.459967,-0.292096,0.529169,-0.046674,-0.088682,-0.062346,-0.232455,-0.549852,-0.347034,-0.62804,-0.14845,0.005361,0.019654,-0.624737,1.159491,-1.710221,1.245836,0.2428,1.227492,-0.077206,0.494423,-0.697413,1.418575,-0.577276,-0.587751,0.895239,…,0.124402,-0.38352,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,"""Bei Ehepaaren ist die Höhe der…","""FAVOR""","""gender""","""Männlich""",null,null,null,false,"""llama_4_scout""","""de""","""Männlich""",null,null,null,null,null,null,32216,"""Welfare state & family""","""SD"""
"""Die Begrenzung der Rentenhöhe …","""2025-07-17T17:23:04.951396543Z""","""de_1""",-0.750142,-0.779247,-0.469683,-0.340281,-0.739473,0.213027,-0.758235,0.563675,-0.717392,0.277887,-0.816556,2.919519,-0.088682,0.521029,-0.232455,-0.549852,0.193655,-0.62804,-0.14845,0.631142,0.440315,-0.322711,0.654022,-0.649782,0.410251,-0.082656,-0.491953,-0.077206,-0.616245,1.571582,-0.428613,-0.577276,1.697108,-1.046768,…,0.450662,-0.38352,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,"""Bei Ehepaaren ist die Höhe der…","""FAVOR""","""age""","""18-34""",null,null,null,false,"""llama_4_scout""","""de""",null,"""18-34""",null,null,null,null,null,32216,"""Welfare state & family""","""SD"""
"""Ich denke, die Begrenzung der …","""2025-07-17

# Save the processed full dataset

In [54]:
df["prompt_id"].value_counts().describe()

statistic,prompt_id,count
str,str,f64
"""count""","""32164""",32165.0
"""null_count""","""1""",0.0
"""mean""",null,12.069641
"""std""",null,71.074897
"""min""","""de_0""",6.0
"""25%""",null,12.0
"""50%""",null,12.0
"""75%""",null,12.0
"""max""","""it_999""",12758.0


In [57]:
df.filter(pl.col("prompt_id") == "it_5399", pl.col("model") == "occiglot-7b-eu5-instruct")

argument,argument_id,prompt_id,raw_sequence_length,n_tokens,n_sentences,tokens_per_sentence,n_characters,avg_word_length,n_types,n_long_words,n_lemmas,n_VERB_VerbForm_Fin,n_VERB_VerbForm_Inf,n_VERB_VerbForm_Part,n_VERB_Mood_Imp,n_VERB_Mood_Ind,n_VERB_Mood_Sub,n_VERB_Tense_Past,n_VERB_Tense_Pres,n_VERB_Person_1,n_VERB_Person_2,n_VERB_Person_3,n_VERB_Number_Plur,n_VERB_Number_Sing,n_NOUN_Gender_Fem,n_NOUN_Gender_Masc,n_NOUN_Number_Plur,n_NOUN_Number_Sing,n_PRON_PronType_Dem,n_PRON_PronType_Int,n_PRON_PronType_Prs,n_PRON_PronType_Rel,n_PRON_Gender_Fem,n_PRON_Gender_Masc,n_PRON_Number_Plur,n_PRON_Number_Sing,…,n_entities,n_org,n_gpe,n_person,n_money,n_product,n_time,n_percent,n_work_of_art,n_quantity,n_norp,n_loc,n_event,n_ordinal,n_fac,n_law,n_language,question,stance,sociodemographic_info,sociodemographic_group,info,pro,contra,context,model,language,gender,age,education,civil_status,denomination,residence,political_spectrum,ID_question,topic,condition
str,str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,…,f64,f64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,str,str,str,str,str,str,str,bool,str,str,str,str,str,str,str,str,str,i64,str,str
"""Sono favorevole al tema politi…","""2025-09-12T08:09:18.990795612""","""it_5399""",-0.264615,-0.234494,-0.307621,0.352998,-0.26794,-0.080703,0.115621,0.123806,0.215775,-0.329945,0.116656,0.204349,-0.088682,-0.101084,-0.232455,0.910504,-0.382938,1.965614,-0.14845,-1.300616,-0.858245,0.779147,-0.345398,2.454413,1.143743,-0.035974,-0.491953,-0.077206,-0.616245,-0.697413,-0.428613,-0.577276,-0.587751,-1.046768,…,-0.556488,-0.38352,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,"""Secondo Lei, d’ora in poi dovr…","""FAVOR""","""civil_status""","""Ledig""","""La Confederazione si impegna a…","""Tali misure avrebbero il vanta…","""Gli oppositori di tali misure …",true,"""occiglot-7b-eu5-instruct""","""it""",null,null,null,"""Ledig""",null,null,null,32253,"""Nature conservation""","""C + SD + Ex + P&C"""
"""Sono a favore del tema politic…","""2025-09-12T08:10:02.522946835""","""it_5399""",-0.250786,-0.304784,-0.388652,0.75741,-0.246507,0.07903,0.142102,0.848297,0.11924,-0.239617,0.623377,1.492997,-0.088682,-0.432364,-0.232455,2.599935,-0.297252,-0.62804,-0.14845,0.062978,0.975016,0.499943,-0.601933,1.28396,1.666578,-1.240544,0.704916,-0.077206,-0.229689,1.438112,-0.428613,0.375466,-0.587751,-0.145575,…,-0.556488,-0.38352,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,"""Secondo Lei, d’ora in poi dovr…","""FAVOR""","""civil_status""","""Ledig""","""La Confederazione si impegna a…","""Tali misure avrebbero il vanta…","""Gli oppositori di tali misure …",true,"""occiglot-7b-eu5-instruct""","""it""",null,null,null,"""Ledig""",null,null,null,32253,"""Nature conservation""","""C + SD + Ex + P&C"""
"""Sono favorevole a questo tema …","""2025-09-12T08:10:47.339323997""","""it_5399""",-0.639516,-0.682597,-0.550714,1.219595,-0.634092,0.180631,-0.413989,1.011574,-0.459967,2.097044,-1.606196,0.128829,-0.088682,0.185277,9.927345,0.811497,1.91932,0.983835,-0.14845,1.842584,3.367578,0.261978,-1.04761,2.60343,0.933129,-0.213206,1.57721,-0.077206,-0.616245,1.148549,-0.428613,1.069837,-0.587751,-0.267771,…,-0.556488,-0.38352,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,"""Secondo Lei, d’ora in poi dovr…","""FAVOR""","""civil_status""","""Ledig""","""La Confederazione si impegna a…","""Tali misure avrebbero il vanta…","""Gli oppositori di tali misure …",true,"""occiglot-7b-eu5-instruct""","""it""",null,null,null,"""Ledig""",null,null,null,32253,"""Nature conservation""","""C + SD + Ex + P&C"""
"""C ""","""2025-09-12T10:06:21.290339470""","""it_5399""",-1.244888,-1.192205,-0.631745,-2.073476,-1.244941,-1.331794,-1.817455,-3.530497,-1.907983,-1.481626,-1.606196,-0.909567,-0.088682,-1.279825,-0.232455,-0.549852,-1.47543,-0.62804,-0.14845,-1.300616,-0.858245,-1.255052,-2.525951,-1.710221,-1.962812,-2.650146,-0.491953,-0.077206,-0.616245,-0.697413,-0.428613,-0.57727

In [58]:
df.filter(pl.col("prompt_id") == "it_5398", pl.col("model") == "occiglot-7b-eu5-instruct")

argument,argument_id,prompt_id,raw_sequence_length,n_tokens,n_sentences,tokens_per_sentence,n_characters,avg_word_length,n_types,n_long_words,n_lemmas,n_VERB_VerbForm_Fin,n_VERB_VerbForm_Inf,n_VERB_VerbForm_Part,n_VERB_Mood_Imp,n_VERB_Mood_Ind,n_VERB_Mood_Sub,n_VERB_Tense_Past,n_VERB_Tense_Pres,n_VERB_Person_1,n_VERB_Person_2,n_VERB_Person_3,n_VERB_Number_Plur,n_VERB_Number_Sing,n_NOUN_Gender_Fem,n_NOUN_Gender_Masc,n_NOUN_Number_Plur,n_NOUN_Number_Sing,n_PRON_PronType_Dem,n_PRON_PronType_Int,n_PRON_PronType_Prs,n_PRON_PronType_Rel,n_PRON_Gender_Fem,n_PRON_Gender_Masc,n_PRON_Number_Plur,n_PRON_Number_Sing,…,n_entities,n_org,n_gpe,n_person,n_money,n_product,n_time,n_percent,n_work_of_art,n_quantity,n_norp,n_loc,n_event,n_ordinal,n_fac,n_law,n_language,question,stance,sociodemographic_info,sociodemographic_group,info,pro,contra,context,model,language,gender,age,education,civil_status,denomination,residence,political_spectrum,ID_question,topic,condition
str,str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,…,f64,f64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,str,str,str,str,str,str,str,bool,str,str,str,str,str,str,str,str,str,i64,str,str
"""È imperativo che le aziende ag…","""2025-09-12T08:09:18.990777969""","""it_5398""",-0.696365,-0.708956,-0.550714,1.046276,-0.69482,0.087721,-0.625833,0.457333,-0.588679,0.026528,-0.252527,2.372507,-0.088682,-0.50803,-0.232455,3.752983,-0.044786,-0.62804,-0.14845,0.355177,2.48091,0.343247,-0.189644,1.471097,1.596782,-0.816225,1.688058,-0.077206,-0.616245,1.24744,-0.428613,1.158075,-0.587751,-0.226039,…,-0.556488,-0.38352,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,"""Secondo Lei, d’ora in poi dovr…","""FAVOR""","""age""","""18-34""","""La Confederazione si impegna a…","""Tali misure avrebbero il vanta…","""Gli oppositori di tali misure …",true,"""occiglot-7b-eu5-instruct""","""it""",null,"""18-34""",null,null,null,null,null,32253,"""Nature conservation""","""C + SD + Ex + P&C"""
"""""Come giovane cittadino svizze…","""2025-09-12T08:10:02.522930621""","""it_5398""",-0.578056,-0.559588,-0.388652,-0.080301,-0.580509,-0.040557,-0.255106,0.507591,-0.266898,0.253784,0.470666,-0.909567,-0.088682,0.49636,-0.232455,-0.549852,0.170791,0.674709,-0.14845,-0.030419,-0.004398,-0.02896,-0.733715,1.776155,0.767836,-0.680565,-0.491953,-0.077206,-0.076126,0.794529,-0.428613,-0.577276,-0.587751,-1.046768,…,-0.22537,-0.38352,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,"""Secondo Lei, d’ora in poi dovr…","""FAVOR""","""age""","""18-34""","""La Confederazione si impegna a…","""Tali misure avrebbero il vanta…","""Gli oppositori di tali misure …",true,"""occiglot-7b-eu5-instruct""","""it""",null,"""18-34""",null,null,null,null,null,32253,"""Nature conservation""","""C + SD + Ex + P&C"""
"""Sostengo l'idea di riservare p…","""2025-09-12T08:10:47.339307547""","""it_5398""",-0.616468,-0.647452,-0.469683,0.237451,-0.610873,0.12949,-0.49343,0.864978,-0.459967,1.199536,0.800327,1.035366,-0.088682,0.778294,4.524912,1.999976,0.432096,0.881494,-0.14845,0.907108,2.109892,0.876013,0.589125,0.309663,0.74926,-0.693963,1.445834,-0.077206,-0.616245,1.031346,-0.428613,0.965258,-0.587751,-0.317231,…,0.210864,-0.38352,0,0,0,0,0,0,0,0,0,2,0,0,0,0,0,"""Secondo Lei, d’ora in poi dovr…","""FAVOR""","""age""","""18-34""","""La Confederazione si impegna a…","""Tali misure avrebbero il vanta…","""Gli oppositori di tali misure …",true,"""occiglot-7b-eu5-instruct""","""it""",null,"""18-34""",null,null,null,null,null,32253,"""Nature conservation""","""C + SD + Ex + P&C"""


In [52]:
df.write_csv("full_corpus.csv")